<div dir=rtl> 

# تدريب موديل التنبؤ بالأمراض 

موديل XGBoost واحد يتنبأ بالمرض من قائمة أعراض عربية — يُستخدَم كإشارة تعزيزية اختيارية بـ
`nodes/ml_corroborate.py` 

## المصدر
لكل مرض من الـ49 بقاعدة معرفة Healix (`rag/knowledge_base/*.json`)، نجمع اتحاد (union) مصدرين
عربيين حقيقيين:
1. أعراض Healix الأصلية — موثّقة WHO/CDC/NICE.
2. أعراض ويب طب (`scripts/webteb_output/webteb_candidates.json`، من `scripts/scrape_webteb.py`).

ثم **نفلتر لمفردات `vocabulary/symptoms.py` القياسية حصرًا** — أي عبارة (من أي مصدر) غير مطابقة
حرفيًا (بعد `normalize`) لمصطلح قياسي موجود تُستبعد. 
`ml/feature_mapper.py`) — استبعاد بسيط. **الفائدة**: كل عمود بالموديل النهائي قابل للاستخدام
100% وقت التشغيل الحقيقي (`nodes/extract_symptoms.py` ما بينتج غير أسماء قياسية أصلًا) — بعكس
## طريقة التوليد
بما إنه ما عنا "مرضى حقيقيين" كتار، نولّد أمثلة تدريب من كل التركيبات الفرعية الممكنة لقائمة أعراض
كل مرض (بحد أدنى **2 عرَض** لكل تركيبة — نفس `MIN_MATCHED_SYMPTOMS`/`MIN_REQUIRED_MATCHED`
بالإنتاج الحقيقي).

## المخرج
حزمة إنتاج كاملة تُحفظ مباشرة بـ `ml/models/xgboost-healix-arabic-v1.0/` (بالصيغة يلي
`ml/model_loader.py` يتوقعها بالضبط)، بالإضافة لـ `density_floor_export.json` (مدخل تحديث
`ml/density_floor.py`).
</div>

## 1. الإعدادات والاستيرادات

In [32]:
import glob
import hashlib
import itertools
import json
import os
import shutil
import sys

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from xgboost import XGBClassifier

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = r"C:\healix-ai-medical-assistant"
sys.path.insert(0, PROJECT_ROOT)

from rules.crisis import normalize
from vocabulary.symptoms import CANONICAL_SYMPTOMS

RAG_KB_DIR = os.path.join(PROJECT_ROOT, "rag", "knowledge_base")
WEBTEB_CANDIDATES_PATH = os.path.join(PROJECT_ROOT, "scripts", "webteb_output", "webteb_candidates.json")

MODEL_VERSION = "xgboost-healix-arabic-v1.0"
BUNDLE_DIR = os.path.join(PROJECT_ROOT, "ml", "models", MODEL_VERSION)

MIN_SYMPTOMS = 2          # نفس MIN_MATCHED_SYMPTOMS / MIN_REQUIRED_MATCHED بالإنتاج
MAX_EXAMPLES_PER_DISEASE = 60  # سقف يمنع مرض كتير الأعراض من الهيمنة على التوازن الطبقي


## 2. تحميل ودمج مصادر الأعراض (Healix KB + ويب طب)

In [ ]:
# كل أمراض قاعدة معرفة Healix (49) + أعراضها العربية الأصلية
rag_kb = {}
for fpath in sorted(glob.glob(os.path.join(RAG_KB_DIR, "*.json"))):
    with open(fpath, encoding="utf-8") as f:
        entry = json.load(f)
    rag_kb[entry["name"]] = entry["symptoms"]

# أعراض ويب طب
webteb_symptoms_by_disease = {}
if os.path.exists(WEBTEB_CANDIDATES_PATH):
    with open(WEBTEB_CANDIDATES_PATH, encoding="utf-8") as f:
        webteb_candidates = json.load(f)
    webteb_symptoms_by_disease = {
        c["healix_target_name"]: c["symptoms"] for c in webteb_candidates if c["symptoms"]
    }

# اتحاد (union) المصدرين لكل مرض من الـ49 كاملين
combined_symptoms = {}
for name, healix_syms in rag_kb.items():
    extra = webteb_symptoms_by_disease.get(name, [])
    combined_symptoms[name] = list(dict.fromkeys(healix_syms + extra))

n_with_webteb = sum(1 for n in rag_kb if n in webteb_symptoms_by_disease)
print(f"إجمالي أمراض قاعدة المعرفة: {len(rag_kb)}")
print(f"منهم عندهم إضافة أعراض حقيقية من ويب طب: {n_with_webteb}")
print(f"الباقي (بمصدر Healix KB لحالهم فقط): {len(rag_kb) - n_with_webteb}")
65

إجمالي أمراض قاعدة المعرفة: 49
منهم عندهم إضافة أعراض حقيقية من ويب طب: 35
الباقي (بمصدر Healix KB لحالهم فقط): 14


## 3. فلترة لمفردات Healix القياسية حصرًا
<div dir=rtl>
أي عبارة (من Healix أو ويب طب) غير مطابقة حرفيًا (بعد `normalize`) لمصطلح بـ
`vocabulary/symptoms.py` تُستبعد — بلا أي تخمين ترجمة.

In [ ]:
_canonical_n = {normalize(s): s for s in CANONICAL_SYMPTOMS}

def filter_to_canonical(symptoms):
    """بس العناصر المطابقة حرفيًا (بعد normalize) لمصطلح قياسي — الصياغة
    المُرجَعة هي الصياغة القياسية المعتمدة، مو صياغة المصدر الأصلي."""
    out, seen = [], set()
    for s in symptoms:
        n = normalize(s)
        if n in _canonical_n and n not in seen:
            out.append(_canonical_n[n])
            seen.add(n)
    return out

combined_symptoms_final = {
    name: filter_to_canonical(syms) for name, syms in combined_symptoms.items()
}

too_thin_final = [
    name for name, syms in combined_symptoms_final.items() if len(syms) < MIN_SYMPTOMS
]

print("عدد الأعراض القياسية فقط (بعد الفلترة) لكل مرض:")
for name, syms in sorted(combined_symptoms_final.items(), key=lambda kv: len(kv[1])):
    flag = " أقل من الحد الأدنى" if len(syms) < MIN_SYMPTOMS else ""
    print(f"  {len(syms):2d}  {name}{flag}")

if too_thin_final:
    print(f"\n أمراض بأقل من {MIN_SYMPTOMS} أعراض قياسية — لن تُدرَّب لها أي أمثلة إطلاقًا:")
    for name in too_thin_final:
        print(f"  - {name}")


عدد الأعراض القياسية فقط (بعد الفلترة) لكل مرض:
   2  Cutaneous Leishmaniasis
   2  Hypertension
   2  Pediculosis Capitis
   2  Roseola
   2  Scabies
   3  Acute Musculoskeletal Strain
   3  Acute Otitis Media
   3  Bacterial Vaginosis
   3  Benign Paroxysmal Positional Vertigo
   3  Dysmenorrhea
   3  Gout
   3  Pinworm Infection
   3  Tendinitis
   3  Tension-Type Headache
   3  Urticaria
   3  Vaginal Candidiasis
   4  Asthma
   4  Atopic Dermatitis
   4  Chickenpox
   4  Gastroesophageal Reflux Disease
   4  Hand, Foot, and Mouth Disease
   4  Infectious Mononucleosis
   4  Irritable Bowel Syndrome
   4  Kidney Stones
   4  Migraine
   4  Mumps
   4  Otitis Externa
   4  Peptic Ulcer Disease
   4  Polycystic Ovary Syndrome
   4  Rheumatoid Arthritis
   4  Rubella
   4  Streptococcal Pharyngitis
   4  Tonsillitis
   4  Typhoid Fever
   4  Urinary Tract Infection
   4  Viral Pharyngitis
   5  Acute Bronchitis
   5  Acute Gastroenteritis
   5  Allergic Rhinitis
   5  Community-Acquir

## 4. توليد أمثلة تدريب صناعية (كل التركيبات الفرعية الممكنة)

In [35]:
def generate_symptom_combinations(symptoms, min_symptoms, max_examples, rng):
    """كل التركيبات الفرعية من قائمة أعراض مرض واحد، بحد أدنى min_symptoms.
    لو عدد التركيبات أكبر من max_examples، تُؤخذ عيّنة عشوائية (بذرة ثابتة)
    بدل كلهم — لمنع مرض كتير الأعراض من الهيمنة على التوازن الطبقي."""
    n = len(symptoms)
    all_combos = []
    for r in range(min_symptoms, n + 1):
        all_combos.extend(itertools.combinations(symptoms, r))
    if len(all_combos) > max_examples:
        idx = rng.choice(len(all_combos), size=max_examples, replace=False)
        all_combos = [all_combos[i] for i in sorted(idx)]
    return all_combos


rng = np.random.RandomState(RANDOM_SEED)
synthetic_rows = []
for disease_name, symptoms in combined_symptoms_final.items():
    if len(symptoms) < MIN_SYMPTOMS:
        continue
    for combo in generate_symptom_combinations(symptoms, MIN_SYMPTOMS, MAX_EXAMPLES_PER_DISEASE, rng):
        synthetic_rows.append({"disease": disease_name, "symptom_set": frozenset(combo)})

symptom_vocab = sorted({s for row in synthetic_rows for s in row["symptom_set"]})
print(f"إجمالي الأمثلة: {len(synthetic_rows)}")
print(f"إجمالي الأعمدة (كلها مفردات Healix القياسية، قابلة للاستخدام 100%): {len(symptom_vocab)}")

def build_binary_row(symptom_set, vocab):
    return {s: (1 if s in symptom_set else 0) for s in vocab}

binary_rows = [build_binary_row(r["symptom_set"], symptom_vocab) for r in synthetic_rows]
feature_matrix = pd.DataFrame(binary_rows, columns=symptom_vocab)
feature_matrix.insert(0, "disease", [r["disease"] for r in synthetic_rows])

print("شكل مصفوفة الميزات:", feature_matrix.shape)
feature_matrix.head()


إجمالي الأمثلة: 737
إجمالي الأعمدة (كلها مفردات Healix القياسية، قابلة للاستخدام 100%): 98
شكل مصفوفة الميزات: (737, 99)


,disease,احتقان انف,احساس بحركه بفروه الراس,احمرار,احمرار العين,ارتجاع حمضي,ازيز صدر,اسهال,اضطراب الدوره الشهريه,اضطراب نوم,...,عطش شديد,غثيان,فقدان الوعي,فقدان حاسه الشم,فقدان سمع,فقدان شهيه,قشور عسليه اللون على الجلد,نقص وزن غير مبرر,نمو شعر زائد,يرقان
0,Acute Bronchitis,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Acute Bronchitis,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Acute Bronchitis,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Acute Bronchitis,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Acute Bronchitis,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 5. تقسيم البيانات (Train / Validation / Test)

In [ ]:
X_all = feature_matrix[symptom_vocab]
y_all = feature_matrix["disease"]

class_counts = y_all.value_counts()
rare_classes = class_counts[class_counts < 3].index.tolist()
if rare_classes:
    print(" أمراض بأقل من 3 أمثلة (بالكامل بـ Train، بلا تقييم موضوعي ممكن):")
    print(class_counts[class_counts < 3])

rare_mask = y_all.isin(rare_classes)
X_common, y_common = X_all[~rare_mask], y_all[~rare_mask]
X_rare, y_rare = X_all[rare_mask], y_all[rare_mask]

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_common, y_common, test_size=0.15, stratify=y_common, random_state=RANDOM_SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.15 / 0.85,
    stratify=y_train_val, random_state=RANDOM_SEED,
)
X_train = pd.concat([X_train, X_rare])
y_train = pd.concat([y_train, y_rare])

print(f"\nTrain: {len(X_train)} | Validation: {len(X_val)} | Test: {len(X_test)}")
print(f"عدد الأمراض المُمثَّلة — Train: {y_train.nunique()} | Val: {y_val.nunique()} | Test: {y_test.nunique()} / {y_all.nunique()} إجمالي")


⚠️ أمراض بأقل من 3 أمثلة (بالكامل بـ Train، بلا تقييم موضوعي ممكن):
disease
Cutaneous Leishmaniasis    1
Hypertension               1
Pediculosis Capitis        1
Roseola                    1
Scabies                    1
Name: count, dtype: int64

Train: 517 | Validation: 110 | Test: 110
عدد الأمراض المُمثَّلة — Train: 49 | Val: 44 | Test: 33 / 49 إجمالي


## 6. تدريب XGBoost وتقييمه

In [37]:
label_encoder = LabelEncoder()
label_encoder.fit(y_all)

y_train_enc = label_encoder.transform(y_train)
y_val_enc = label_encoder.transform(y_val)
y_test_enc = label_encoder.transform(y_test)

model = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1, subsample=0.9,
    colsample_bytree=0.8, objective="multi:softprob", eval_metric="mlogloss",
    random_state=RANDOM_SEED, n_jobs=-1,
)
model.fit(X_train, y_train_enc, eval_set=[(X_val, y_val_enc)], verbose=False)

print("تمّ التدريب بنجاح.")
print("عدد الميزات:", model.n_features_in_)
print("عدد الفئات:", len(label_encoder.classes_))

y_pred_enc = model.predict(X_test)
accuracy = accuracy_score(y_test_enc, y_pred_enc)
macro_f1 = f1_score(y_test_enc, y_pred_enc, average="macro", zero_division=0)
weighted_f1 = f1_score(y_test_enc, y_pred_enc, average="weighted", zero_division=0)

print(f"\nTest Accuracy:    {accuracy:.4f}")
print(f"Test Macro F1:    {macro_f1:.4f}")
print(f"Test Weighted F1: {weighted_f1:.4f}")

print("\n" + classification_report(
    label_encoder.inverse_transform(y_test_enc),
    label_encoder.inverse_transform(y_pred_enc),
    zero_division=0,
))


تمّ التدريب بنجاح.
عدد الميزات: 98
عدد الفئات: 49

Test Accuracy:    0.8545
Test Macro F1:    0.7433
Test Weighted F1: 0.8620

                                      precision    recall  f1-score   support

                    Acute Bronchitis       0.80      1.00      0.89         4
               Acute Gastroenteritis       1.00      0.75      0.86         4
                     Acute Sinusitis       1.00      1.00      1.00         8
                   Allergic Rhinitis       1.00      1.00      1.00         4
                              Asthma       1.00      1.00      1.00         2
                   Atopic Dermatitis       0.50      1.00      0.67         2
Benign Paroxysmal Positional Vertigo       0.00      0.00      0.00         0
                          Chickenpox       1.00      0.50      0.67         2
        Community-Acquired Pneumonia       1.00      0.25      0.40         4
                      Conjunctivitis       1.00      1.00      1.00         8
              

## 7. بناء حزمة الإنتاج

تُحفظ مباشرة بـ `ml/models/xgboost-healix-arabic-v1.0/` بالصيغة يلي `ml/model_loader.py` يتحقق
منها بالضبط، بالإضافة لـ `density_floor_export.json` (مدخل تحديث `ml/density_floor.py`).

In [38]:
if os.path.exists(BUNDLE_DIR):
    shutil.rmtree(BUNDLE_DIR)
os.makedirs(BUNDLE_DIR)

joblib.dump(model, os.path.join(BUNDLE_DIR, "model.joblib"))
joblib.dump(label_encoder, os.path.join(BUNDLE_DIR, "label_encoder.joblib"))

feature_schema = {
    "schema_version": "healix-arabic-v1.0",
    "layers": {"model_feature_order": symptom_vocab},
}
with open(os.path.join(BUNDLE_DIR, "feature_schema.json"), "w", encoding="utf-8") as f:
    json.dump(feature_schema, f, ensure_ascii=False, indent=2)

metadata = {
    "model_name": "xgboost",
    "model_version": MODEL_VERSION,
    "algorithm": "XGBClassifier",
    "dataset_version": "healix-kb-plus-webteb-canonical-only-v1",
    "feature_schema_version": "healix-arabic-v1.0",
    "n_features": int(model.n_features_in_),
    "n_classes": int(len(label_encoder.classes_)),
    "evaluation_metrics": {
        "holdout_test": {
            "accuracy": float(accuracy),
            "macro_f1": float(macro_f1),
            "weighted_f1": float(weighted_f1),
        }
    },
    "known_limitations": [
        "Synthetic symptom-combination data (union of rag/knowledge_base/*.json and "
        "webteb.com, filtered to vocabulary/symptoms.py canonical terms only) -- NOT "
        "real patient data.",
        f"Classes with <3 examples (no held-out test evaluation): {rare_classes}",
        f"Diseases excluded entirely (fewer than {MIN_SYMPTOMS} canonical symptoms "
        f"available): {too_thin_final}",
        "Every feature column IS producible by nodes/extract_symptoms.py at real "
        "runtime (canonical vocabulary only) -- unlike the previous English-Kaggle "
        "bundle where only 52/131 columns were ever mappable.",
    ],
}
with open(os.path.join(BUNDLE_DIR, "metadata.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

checksums = {}
for fname in ("model.joblib", "label_encoder.joblib", "feature_schema.json", "metadata.json"):
    checksums[fname] = hashlib.sha256(open(os.path.join(BUNDLE_DIR, fname), "rb").read()).hexdigest()
with open(os.path.join(BUNDLE_DIR, "checksums.json"), "w", encoding="utf-8") as f:
    json.dump(checksums, f, indent=2)

# لكل مرض، الأعمدة القياسية يلي فعليًا ظهرت بأي مثال تدريب/اختبار له —
# مدخل لتحديث ml/density_floor.py's EXPECTED_FEATURES_BY_DISEASE.
density_floor_export = {}
for disease in label_encoder.classes_:
    disease_rows = feature_matrix[feature_matrix["disease"] == disease]
    density_floor_export[disease] = [c for c in symptom_vocab if disease_rows[c].sum() > 0]

with open(os.path.join(BUNDLE_DIR, "density_floor_export.json"), "w", encoding="utf-8") as f:
    json.dump(density_floor_export, f, ensure_ascii=False, indent=2)

print("✓ حزمة الإنتاج جاهزة:", BUNDLE_DIR)
print("\nكل الأمراض المشمولة:")
for c in sorted(label_encoder.classes_):
    print(" -", c)


✓ حزمة الإنتاج جاهزة: C:\healix-ai-medical-assistant\ml\models\xgboost-healix-arabic-v1.0

كل الأمراض المشمولة:
 - Acute Bronchitis
 - Acute Gastroenteritis
 - Acute Musculoskeletal Strain
 - Acute Otitis Media
 - Acute Sinusitis
 - Allergic Rhinitis
 - Asthma
 - Atopic Dermatitis
 - Bacterial Vaginosis
 - Benign Paroxysmal Positional Vertigo
 - Chickenpox
 - Community-Acquired Pneumonia
 - Conjunctivitis
 - Cutaneous Leishmaniasis
 - Dysmenorrhea
 - Gastroesophageal Reflux Disease
 - Gout
 - Hand, Foot, and Mouth Disease
 - Hepatitis A
 - Herpes Zoster
 - Hypertension
 - Impetigo
 - Infectious Mononucleosis
 - Influenza
 - Iron Deficiency Anaemia
 - Irritable Bowel Syndrome
 - Kidney Stones
 - Measles
 - Migraine
 - Mumps
 - Otitis Externa
 - Pediculosis Capitis
 - Peptic Ulcer Disease
 - Pinworm Infection
 - Polycystic Ovary Syndrome
 - Rheumatoid Arthritis
 - Roseola
 - Rubella
 - Scabies
 - Streptococcal Pharyngitis
 - Tendinitis
 - Tension-Type Headache
 - Tonsillitis
 - Type 2 Di